# Segmentacion Automatica de Columna Vertebral y Vertebras
## Proyecto Final - Maestria en Inteligencia Artificial
### Universidad de los Andes

---

**Este notebook contiene todo el pipeline del proyecto:**
1. Configuracion del entorno
2. Carga y exploracion del dataset
3. Entrenamiento de modelos (o carga de pesos pre-entrenados)
4. Evaluacion y visualizacion de resultados
5. Calculo del angulo de Cobb
6. Comparacion de modelos

**IMPORTANTE para compañeros del equipo:**
- Si ya se tienen los pesos entrenados en `checkpoints/`, NO es necesario reentrenar
- Ir directo a la **Seccion 4** para cargar pesos y ver resultados
- Este notebook funciona en **CPU** para inferencia (no necesitan GPU)

---
## 1. Configuracion del Entorno

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# Nuestros modulos
from spine_segmentation.config import *
from spine_segmentation.data.dataset import SpineBinaryDataset, SpineMulticlassDataset, get_dataloaders
from spine_segmentation.data.splits import load_splits
from spine_segmentation.data.class_mapping import get_class_names, get_num_classes
from spine_segmentation.data.transforms import denormalize_image
from spine_segmentation.models.smp_models import create_model
from spine_segmentation.models.losses import get_loss_function, compute_class_weights
from spine_segmentation.training.trainer import Trainer
from spine_segmentation.evaluation.metrics import compute_test_metrics, print_metrics_table
from spine_segmentation.evaluation.visualize import (
    create_binary_overlay, create_multiclass_overlay,
    plot_per_class_dice, save_prediction_grid
)
from spine_segmentation.evaluation.cobb_angle import (
    cobb_from_binary, cobb_from_multiclass,
    evaluate_cobb_angles, plot_cobb_comparison, load_ground_truth_cobb_angles
)
from spine_segmentation.postprocessing.morphology import clean_binary_mask, clean_multiclass_mask

plt.rcParams['figure.dpi'] = 120

# Detectar dispositivo
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('Ejecutando en CPU - suficiente para inferencia con pesos pre-entrenados')

---
## 2. Carga y Exploracion del Dataset

In [ ]:
# Cargar indice del dataset
df = pd.read_csv(DATASET_INDEX_CSV)
print(f'Total de muestras: {len(df)}')
print(f'\nDistribucion:')
print(df['split'].value_counts())

# Cargar splits (train/val/test)
splits = load_splits()

# Mostrar distribucion por split
for split_name, images in splits.items():
    split_df = df[df['image'].isin(images)]
    counts = split_df['split'].value_counts()
    print(f'{split_name:>6}: {len(images):3d} imagenes | '
          f'Normal={counts.get("Normal", 0)} | '
          f'Escoliosis={counts.get("Scoliosis", 0)}')

In [ ]:
# Visualizar muestras del dataset con sus mascaras
sample_indices = [0, 71, 100, 150, 200, 240]  # Mix de normales y escoliosis
fig, axes = plt.subplots(3, 6, figsize=(24, 12))

for i, idx in enumerate(sample_indices):
    if idx >= len(df):
        continue
    row = df.iloc[idx]
    
    # Imagen original
    img = cv2.imread(str(DATASET_ROOT / row['radiograph_path']))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"{row['image']} ({row['split']})", fontsize=9)
    axes[0, i].axis('off')
    
    # Mascara binaria
    mask_bin = cv2.imread(str(DATASET_ROOT / row['label_binary_path']), cv2.IMREAD_GRAYSCALE)
    axes[1, i].imshow(mask_bin, cmap='gray')
    axes[1, i].set_title('Mascara Binaria', fontsize=9)
    axes[1, i].axis('off')
    
    # Mascara multiclase (color)
    mask_color = cv2.imread(str(DATASET_ROOT / row['multiclass_color_jpg']))
    if mask_color is not None:
        mask_color = cv2.cvtColor(mask_color, cv2.COLOR_BGR2RGB)
        axes[2, i].imshow(mask_color)
    axes[2, i].set_title('Mascara Multiclase', fontsize=9)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Radiografia', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Segmentacion\nBinaria', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Segmentacion\nMulticlase', fontsize=12, fontweight='bold')
plt.suptitle('Muestras del Dataset MaIA Scoliosis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Entrenamiento de Modelos

**⚠️ NOTA:** Esta seccion requiere GPU y tarda ~30-60 min por modelo.  
**Si ya tienes los pesos entrenados en `checkpoints/`, salta a la Seccion 4.**

### Arquitecturas a entrenar:
| Modelo | Encoder | Tarea | Clases |
|--------|---------|-------|--------|
| M1 | ResNet50 | Binaria | 1 (columna vs fondo) |
| M2 | EfficientNet-B4 | Binaria | 1 |
| M3 | DeepLabV3+ ResNet50 | Binaria | 1 |
| M4 | ResNet50 | Multiclase | 24 (vertebras) |
| M5 | EfficientNet-B4 | Multiclase | 24 |
| M6 | DeepLabV3+ ResNet50 | Multiclase | 24 |

In [ ]:
# ============================================================
# OPCION A: ENTRENAR (requiere GPU, ~30-60 min por modelo)
# ============================================================
# Descomentar y ejecutar SOLO si tienes GPU y quieres entrenar

ENTRENAR = False  # Cambiar a True para entrenar

if ENTRENAR and DEVICE == 'cuda':
    print('Iniciando entrenamiento...')
    
    # --- Modelo 1: U-Net + ResNet50 Binario ---
    print('\n' + '='*60)
    print('MODELO 1: U-Net + ResNet50 - Segmentacion Binaria')
    print('='*60)
    
    loaders_bin = get_dataloaders(task='binary', splits_dict=splits)
    model_bin = create_model('unet_resnet50', num_classes=1)
    loss_fn_bin = get_loss_function('binary')
    
    trainer_bin = Trainer(
        model=model_bin, train_loader=loaders_bin['train'],
        val_loader=loaders_bin['val'], loss_fn=loss_fn_bin,
        task='binary', model_name='unet_resnet50', num_classes=1,
    )
    results_bin = trainer_bin.train()
    
else:
    print('Entrenamiento desactivado. Los pesos pre-entrenados se cargaran en la Seccion 4.')
    print(f'Pesos disponibles en: {CHECKPOINTS_DIR}')
    print(f'Archivos encontrados: {list(CHECKPOINTS_DIR.glob("*.pth"))}')

---
## 4. Carga de Pesos Pre-entrenados y Evaluacion

**Esta seccion NO requiere GPU.** Carga los modelos entrenados y evalua en el conjunto de test.

Los pesos estan guardados en la carpeta `checkpoints/` con el formato:
- `{modelo}_{tarea}_best.pth` (ej: `unet_resnet50_binary_best.pth`)

In [ ]:
def load_trained_model(model_name, task, num_classes, device='cpu'):
    """
    Carga un modelo con pesos pre-entrenados.
    Funciona en CPU - no necesita GPU.
    """
    ckpt_path = CHECKPOINTS_DIR / f'{model_name}_{task}_best.pth'
    
    if not ckpt_path.exists():
        print(f'  Checkpoint no encontrado: {ckpt_path}')
        return None
    
    model = create_model(model_name, num_classes=num_classes)
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    best_dice = checkpoint.get('best_val_dice', 'N/A')
    best_epoch = checkpoint.get('epoch', 'N/A')
    print(f'  Cargado: {ckpt_path.name}')
    print(f'  Mejor Dice validacion: {best_dice:.4f} (epoca {best_epoch + 1})')
    
    return model

# Listar checkpoints disponibles
print('Checkpoints disponibles:')
for f in sorted(CHECKPOINTS_DIR.glob('*.pth')):
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name} ({size_mb:.1f} MB)')

In [ ]:
# Cargar todos los modelos disponibles
modelos = {}

# Modelos binarios
for model_name in ['unet_resnet50', 'unet_efficientnet_b4', 'deeplabv3plus_resnet50']:
    print(f'\nCargando {model_name} (binario)...')
    m = load_trained_model(model_name, 'binary', num_classes=1, device=DEVICE)
    if m is not None:
        modelos[f'{model_name}_binary'] = m

# Modelos multiclase
for model_name in ['unet_resnet50', 'unet_efficientnet_b4', 'deeplabv3plus_resnet50']:
    print(f'\nCargando {model_name} (multiclase 24)...')
    m = load_trained_model(model_name, 'multiclass', num_classes=24, device=DEVICE)
    if m is not None:
        modelos[f'{model_name}_multiclass'] = m

print(f'\nTotal modelos cargados: {len(modelos)}')

### 4.1 Evaluacion en el Conjunto de Test

In [ ]:
# Evaluar cada modelo en el test set
resultados = []

for key, model in modelos.items():
    model_name = '_'.join(key.split('_')[:-1])
    task = key.split('_')[-1]
    
    print(f'\n{"="*60}')
    print(f'Evaluando: {key}')
    print(f'{"="*60}')
    
    if task == 'binary':
        loaders = get_dataloaders(task='binary', splits_dict=splits)
        metrics = compute_test_metrics(model, loaders['test'], task='binary', num_classes=1, device=DEVICE)
    else:
        loaders = get_dataloaders(task='multiclass', scheme='vertebrae_24', splits_dict=splits)
        metrics = compute_test_metrics(model, loaders['test'], task='multiclass', num_classes=24, device=DEVICE)
    
    class_names = get_class_names('vertebrae_24') if task == 'multiclass' else None
    print_metrics_table(metrics, class_names=class_names)
    
    resultados.append({
        'Modelo': model_name,
        'Tarea': task,
        'Dice Mean': metrics.get('test_dice_mean', metrics.get('test_dice', 0)),
        'IoU Mean': metrics.get('test_iou_mean', 0),
        'Pixel Acc': metrics.get('test_pixel_acc', 0),
    })

# Tabla de comparacion
if resultados:
    df_results = pd.DataFrame(resultados)
    print('\n' + '='*60)
    print('TABLA COMPARATIVA DE MODELOS')
    print('='*60)
    print(df_results.to_string(index=False, float_format='%.4f'))

### 4.2 Visualizacion de Predicciones

In [ ]:
# Seleccionar el mejor modelo binario para visualizar
best_binary_key = [k for k in modelos if 'binary' in k]
if best_binary_key:
    best_binary_key = best_binary_key[0]
    model_bin = modelos[best_binary_key]
    
    # Cargar dataset de test
    test_ds = SpineBinaryDataset(split='test', splits_dict=splits)
    
    # Predecir y visualizar 6 muestras
    fig, axes = plt.subplots(6, 3, figsize=(18, 36))
    
    for i in range(min(6, len(test_ds))):
        sample = test_ds[i]
        img_tensor = sample['image'].unsqueeze(0).to(DEVICE)
        mask_gt = sample['mask'].numpy()[0]  # (H, W)
        
        with torch.no_grad():
            pred = torch.sigmoid(model_bin(img_tensor)).cpu().numpy()[0, 0]
        pred_binary = (pred > 0.5).astype(np.uint8)
        pred_clean = clean_binary_mask(pred_binary)
        
        # Desnormalizar imagen para visualizar
        img_vis = denormalize_image(sample['image'])
        
        axes[i, 0].imshow(img_vis)
        axes[i, 0].set_title(f"{sample['image_name']} ({sample['condition']})", fontsize=10)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask_gt, cmap='gray')
        axes[i, 1].set_title('Ground Truth', fontsize=10)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(pred_clean, cmap='gray')
        axes[i, 2].set_title('Prediccion', fontsize=10)
        axes[i, 2].axis('off')
    
    axes[0, 0].set_title('Radiografia', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('Ground Truth', fontsize=12, fontweight='bold')
    axes[0, 2].set_title('Prediccion del Modelo', fontsize=12, fontweight='bold')
    plt.suptitle(f'Segmentacion Binaria - {best_binary_key}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / 'predicciones_binarias.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No hay modelos binarios cargados.')

In [ ]:
# Visualizar predicciones multiclase
best_multi_key = [k for k in modelos if 'multiclass' in k]
if best_multi_key:
    best_multi_key = best_multi_key[0]
    model_multi = modelos[best_multi_key]
    class_names_24 = get_class_names('vertebrae_24')
    
    test_ds_mc = SpineMulticlassDataset(split='test', scheme='vertebrae_24', splits_dict=splits)
    
    fig, axes = plt.subplots(6, 3, figsize=(18, 36))
    
    for i in range(min(6, len(test_ds_mc))):
        sample = test_ds_mc[i]
        img_tensor = sample['image'].unsqueeze(0).to(DEVICE)
        mask_gt = sample['mask'].numpy()  # (H, W)
        
        with torch.no_grad():
            pred_logits = model_multi(img_tensor)
        pred_mask = pred_logits.argmax(dim=1).cpu().numpy()[0]
        pred_clean = clean_multiclass_mask(pred_mask.astype(np.uint8), num_classes=24)
        
        img_vis = denormalize_image(sample['image'])
        
        axes[i, 0].imshow(img_vis)
        axes[i, 0].set_title(f"{sample['image_name']}", fontsize=10)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask_gt, cmap='nipy_spectral', vmin=0, vmax=23)
        axes[i, 1].set_title('Ground Truth', fontsize=10)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(pred_clean, cmap='nipy_spectral', vmin=0, vmax=23)
        axes[i, 2].set_title('Prediccion', fontsize=10)
        axes[i, 2].axis('off')
    
    plt.suptitle(f'Segmentacion Multiclase (24 clases) - {best_multi_key}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / 'predicciones_multiclase.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No hay modelos multiclase cargados.')

---
## 5. Calculo del Angulo de Cobb

Implementamos **dos metodos** para calcular el angulo de Cobb:

- **Metodo A (Binario):** Esqueletizacion + B-spline + puntos de inflexion (replicando semestre anterior)
- **Metodo B (Multiclase):** Orientacion de placas terminales vertebrales (contribucion nueva)

In [ ]:
# Cargar angulos de Cobb ground truth
gt_cobb = load_ground_truth_cobb_angles()
print(f'Angulos de Cobb ground truth: {len(gt_cobb)} casos de escoliosis')
print(f'Rango: {gt_cobb["cobb_angle_deg"].min():.1f} - {gt_cobb["cobb_angle_deg"].max():.1f} grados')
print(f'Media: {gt_cobb["cobb_angle_deg"].mean():.1f} grados')

In [ ]:
# Calcular angulos de Cobb con el modelo binario
if best_binary_key:
    model_bin = modelos[best_binary_key]
    test_ds_bin = SpineBinaryDataset(split='test', splits_dict=splits)
    
    cobb_predictions_binary = {}
    
    print('Calculando angulos de Cobb (Metodo A: Binario)...')
    for i in range(len(test_ds_bin)):
        sample = test_ds_bin[i]
        name = sample['image_name']
        
        if not name.startswith('S_'):  # Solo escoliosis
            continue
        
        img_tensor = sample['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = torch.sigmoid(model_bin(img_tensor)).cpu().numpy()[0, 0]
        
        pred_binary = clean_binary_mask((pred > 0.5).astype(np.uint8))
        result = cobb_from_binary(pred_binary)
        
        if result['success'] and result['cobb_angle_deg'] is not None:
            cobb_predictions_binary[name] = result['cobb_angle_deg']
    
    # Evaluar
    eval_binary = evaluate_cobb_angles(cobb_predictions_binary)
    print(f'\nResultados Metodo A (Binario):')
    print(f'  Muestras evaluadas: {eval_binary["n_samples"]}')
    print(f'  MAE: {eval_binary["mae_deg"]:.1f} grados')
    print(f'  Correlacion: {eval_binary.get("correlation", 0):.3f}')
    print(f'  Dentro de 5 grados: {eval_binary.get("within_5_deg", 0)*100:.1f}%')
    print(f'  Dentro de 10 grados: {eval_binary.get("within_10_deg", 0)*100:.1f}%')
    
    # Grafica
    plot_cobb_comparison(eval_binary, 
                        save_path=str(OUTPUTS_DIR / 'cobb_metodo_binario.png'),
                        title='Angulo de Cobb - Metodo A (Binario)')

In [ ]:
# Calcular angulos de Cobb con el modelo multiclase
if best_multi_key:
    model_multi = modelos[best_multi_key]
    test_ds_mc = SpineMulticlassDataset(split='test', scheme='vertebrae_24', splits_dict=splits)
    
    cobb_predictions_multi = {}
    
    print('Calculando angulos de Cobb (Metodo B: Multiclase)...')
    for i in range(len(test_ds_mc)):
        sample = test_ds_mc[i]
        name = sample['image_name']
        
        if not name.startswith('S_'):
            continue
        
        img_tensor = sample['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred_logits = model_multi(img_tensor)
        
        pred_mask = pred_logits.argmax(dim=1).cpu().numpy()[0].astype(np.uint8)
        pred_clean = clean_multiclass_mask(pred_mask, num_classes=24)
        result = cobb_from_multiclass(pred_clean)
        
        if result['success'] and result['cobb_angle_deg'] is not None:
            cobb_predictions_multi[name] = result['cobb_angle_deg']
    
    # Evaluar
    eval_multi = evaluate_cobb_angles(cobb_predictions_multi)
    print(f'\nResultados Metodo B (Multiclase):')
    print(f'  Muestras evaluadas: {eval_multi["n_samples"]}')
    print(f'  MAE: {eval_multi["mae_deg"]:.1f} grados')
    print(f'  Correlacion: {eval_multi.get("correlation", 0):.3f}')
    print(f'  Dentro de 5 grados: {eval_multi.get("within_5_deg", 0)*100:.1f}%')
    print(f'  Dentro de 10 grados: {eval_multi.get("within_10_deg", 0)*100:.1f}%')
    
    plot_cobb_comparison(eval_multi,
                        save_path=str(OUTPUTS_DIR / 'cobb_metodo_multiclase.png'),
                        title='Angulo de Cobb - Metodo B (Multiclase)')

---
## 6. Comparacion Final y Conclusiones

In [ ]:
# Tabla comparativa final: Semestre Anterior vs Proyecto Final
comparacion = pd.DataFrame([
    {'Aspecto': 'Framework', 'Semestre Anterior': 'TensorFlow/Keras', 'Proyecto Final': 'PyTorch + SMP'},
    {'Aspecto': 'Segmentacion', 'Semestre Anterior': 'Solo binaria', 'Proyecto Final': 'Binaria + Multiclase (24 clases)'},
    {'Aspecto': 'Arquitecturas', 'Semestre Anterior': '1 (U-Net+ResNet50)', 'Proyecto Final': '3+ (U-Net, DeepLabV3+, multiples encoders)'},
    {'Aspecto': 'Input Size', 'Semestre Anterior': '288x288', 'Proyecto Final': '512x512'},
    {'Aspecto': 'Augmentation', 'Semestre Anterior': 'Ninguna', 'Proyecto Final': '8+ transformaciones'},
    {'Aspecto': 'Data Split', 'Semestre Anterior': '90/10 train/val', 'Proyecto Final': '70/15/15 train/val/test estratificado'},
    {'Aspecto': 'Angulo Cobb', 'Semestre Anterior': 'Solo desde esqueleto binario', 'Proyecto Final': 'Binario + Placas vertebrales'},
    {'Aspecto': 'Tracking', 'Semestre Anterior': 'Ninguno', 'Proyecto Final': 'MLflow'},
    {'Aspecto': 'Despliegue', 'Semestre Anterior': 'Ninguno', 'Proyecto Final': 'App web Gradio + Docker'},
])

print('COMPARACION: Semestre Anterior vs Proyecto Final')
print('='*80)
print(comparacion.to_string(index=False))

In [ ]:
# Guardar todos los resultados
if resultados:
    df_final = pd.DataFrame(resultados)
    df_final.to_csv(OUTPUTS_DIR / 'resultados_finales.csv', index=False)
    print(f'Resultados guardados en {OUTPUTS_DIR / "resultados_finales.csv"}')
    print('\n')
    print(df_final.to_string(index=False, float_format='%.4f'))

print('\n\nFiguras guardadas en:', OUTPUTS_DIR)
for f in sorted(OUTPUTS_DIR.glob('*.png')):
    print(f'  {f.name}')

---
## Instrucciones para el Equipo

### Para ejecutar sin GPU (solo inferencia):
1. Instalar dependencias: `pip install -r requirements.txt`
2. Asegurarse de tener la carpeta `checkpoints/` con los archivos `.pth`
3. Ejecutar este notebook desde la **Seccion 4** en adelante
4. Todo corre en **CPU** - no necesitan GPU

### Para la app web (demo):
```bash
python -m spine_segmentation.deployment.app
```
Abre `http://localhost:7860` en el navegador.

### Archivos de pesos importantes:
- `checkpoints/unet_resnet50_binary_best.pth` - Mejor modelo binario
- `checkpoints/unet_resnet50_multiclass_best.pth` - Mejor modelo multiclase
- `data_splits.json` - Division de datos (para reproducibilidad)